# 27. Remove Element

Given an integer array `nums` and an integer `val`, remove all occurrences of `val` in `nums` in-place. The order of the elements may be changed. Then return the number of elements in nums which are not equal to val.

Consider the number of elements in nums which are not equal to val be k, to get accepted, you need to do the following things:

- Change the array nums such that the first k elements of nums contain the elements which are not equal to val. The remaining elements of nums are not important as well as the size of nums.
- Return k.

**Custom Judge:**

The judge will test your solution with the following code:

```java
int[] nums = [...]; // Input array
int val = ...; // Value to remove
int[] expectedNums = [...]; // The expected answer with correct length.
                            // It is sorted with no values equaling val.

int k = removeElement(nums, val); // Calls your implementation

assert k == expectedNums.length;
sort(nums, 0, k); // Sort the first k elements of nums
for (int i = 0; i < actualLength; i++) {
    assert nums[i] == expectedNums[i];
}
```

If all assertions pass, then your solution will be accepted.

**Example 1:**

    Input: nums = [3,2,2,3], val = 3
    Output: 2, nums = [2,2,_,_]
    Explanation: Your function should return k = 2, with the first two elements of nums being 2.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Example 2:**

    Input: nums = [0,1,2,2,3,0,4,2], val = 2
    Output: 5, nums = [0,1,4,0,3,_,_,_]
    Explanation: Your function should return k = 5, with the first five elements of nums containing 0, 0, 1, 3, and 4.
    Note that the five elements can be returned in any order.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Constraints:**

- `0 <= nums.length <= 100`
- `0 <= nums[i] <= 50`
- `0 <= val <= 100`


---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers (read / write)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(n²) | O(1) |
| 2️⃣ Optimized | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding


🗣️ *"A few quick questions first."*

- Must it be in place? → **Yes**, O(1) extra memory.
- Does the order of the kept elements matter? → **No**, and anything past index `k` is ignored.
- What do I return? → `k`, the number of elements not equal to `val`.
- Edge cases: empty array, every element equals `val`, no element equals `val`.

### Step 1 · Brute force


**Idea:** While `val` is still in the list, delete one copy of it.

🗣️ *"The most direct approach is to keep deleting `val` until it's gone. That's correct, but each `remove` is O(n), because it scans and then shifts everything after it. With up to n copies, that's O(n²)."*

In [1]:
class SolutionBrute:
    def removeElement(self, nums: list[int], val: int) -> int:
        while val in nums:      # O(n) scan
            nums.remove(val)    # O(n) shift
        return len(nums)

### Step 2 · Optimize


**Bottleneck:** Every deletion shifts the whole tail. We move the same elements over and over.

**Key insight:** We don't need to delete anything. We only need the **first `k` slots** to hold the good values. Use a *read* pointer `r` that scans everything and a *write* pointer `k` that marks where the next good value goes.

🗣️ *"Instead of deleting, I'll compact. `r` walks the array and `k` is the next free slot for a keeper. When `nums[r] != val`, I copy it to `nums[k]` and bump `k`. Each element is read once and written at most once, so it's O(n) time and O(1) space."*

In [2]:
class Solution:
    def removeElement(self, nums: list[int], val: int) -> int:
        k = 0                       # next write position
        for r in range(len(nums)):  # read pointer
            if nums[r] != val:
                nums[k] = nums[r]
                k += 1
        return k

### Step 3 · Toy example walkthrough


`nums = [3, 2, 2, 3]`, `val = 3`

| r | nums[r] | keep? | action | k after | nums |
|---|---|---|---|---|---|
| 0 | 3 | ❌ | skip | 0 | [3,2,2,3] |
| 1 | 2 | ✅ | nums[0] = 2 | 1 | [**2**,2,2,3] |
| 2 | 2 | ✅ | nums[1] = 2 | 2 | [2,**2**,2,3] |
| 3 | 3 | ❌ | skip | 2 | [2,2,2,3] |

Return `k = 2`. The first two slots are `[2, 2]` ✅. Whatever sits after them doesn't matter.

In [3]:
def run(cls, nums, val):
    nums = nums[:]
    k = cls().removeElement(nums, val)
    return k, sorted(nums[:k])          # order of kept elements doesn't matter

cases = [
    (([3, 2, 2, 3], 3), (2, [2, 2])),
    (([0, 1, 2, 2, 3, 0, 4, 2], 2), (5, [0, 0, 1, 3, 4])),
    (([], 1), (0, [])),
    (([1, 1, 1], 1), (0, [])),
    (([4, 5], 1), (2, [4, 5])),
]
for args, expected in cases:
    assert run(SolutionBrute, *args) == expected
    assert run(Solution, *args) == expected
    print(args, "->", expected)
print("All tests passed ✅")

([3, 2, 2, 3], 3) -> (2, [2, 2])
([0, 1, 2, 2, 3, 0, 4, 2], 2) -> (5, [0, 0, 1, 3, 4])
([], 1) -> (0, [])
([1, 1, 1], 1) -> (0, [])
([4, 5], 1) -> (2, [4, 5])
All tests passed ✅


### Step 4 · Wrap up


| | Time | Space |
|---|---|---|
| Brute force (repeated remove) | O(n²) | O(1) |
| Optimized (read/write pointers) | O(n) | O(1) |

**Edge cases:** empty array, every element removed, nothing removed.

**Follow-up:** *What if removals are rare and you want fewer writes?* Swap the bad element with the last element and shrink `n`. Order isn't preserved, but writes happen only for removed items.

🗣️ **30-second recap:** *"Deleting in a loop is O(n²) because of the shifting. I compact with a read pointer and a write pointer: every keeper gets copied to the next write slot. That's one pass, O(n) time and O(1) space, and I return the write pointer as `k`."*